# 03 — VAE Training Curves
Plot loss, KL divergence, NDCG@10 and HR@10 across Mult-VAE training epochs. Reads from `data/vae_metrics.jsonl` written by `train_vae.py`.

In [ ]:
import os
os.chdir('..')
from dotenv import load_dotenv
load_dotenv()

import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
metrics_path = Path('./data/vae_metrics.jsonl')
if not metrics_path.exists():
    print('vae_metrics.jsonl not found. Run: python scholar/recsys/train_vae.py')
    # Synthesise dummy data for layout preview
    import numpy as np
    epochs = range(1, 101)
    beta = [min(0.2, e * 0.2 / 20) for e in epochs]
    recon = [2.5 * (0.97**e) + 0.3 for e in epochs]
    kl    = [b * 0.5 * (1 - 0.99**e) for e, b in zip(epochs, beta)]
    ndcg  = [min(0.35, 0.01 * e**0.6) for e in epochs]
    hr    = [min(0.55, 0.015 * e**0.6) for e in epochs]
    df = pd.DataFrame({'epoch': list(epochs), 'recon_loss': recon, 'kl_loss': kl,
                       'ndcg10': ndcg, 'hr10': hr, 'beta': beta})
else:
    rows = [json.loads(l) for l in metrics_path.read_text().splitlines() if l.strip()]
    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} epoch records')

df.head(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Reconstruction loss
axes[0,0].plot(df['epoch'], df['recon_loss'], color='steelblue')
axes[0,0].set_title('Reconstruction loss'); axes[0,0].set_xlabel('Epoch')

# KL loss + beta annealing
ax2 = axes[0,1].twinx()
axes[0,1].plot(df['epoch'], df['kl_loss'], color='darkorange', label='KL loss')
ax2.plot(df['epoch'], df['beta'], color='gray', linestyle='--', label='β')
axes[0,1].set_title('KL loss + β annealing')
axes[0,1].set_xlabel('Epoch')
axes[0,1].legend(loc='upper left'); ax2.legend(loc='upper right')

# NDCG@10
axes[1,0].plot(df['epoch'], df['ndcg10'], color='green')
axes[1,0].set_title('NDCG@10 (val)'); axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylim(0, 0.5)

# HR@10
axes[1,1].plot(df['epoch'], df['hr10'], color='purple')
axes[1,1].set_title('HR@10 (val)'); axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylim(0, 0.7)

plt.suptitle('Mult-VAE training curves', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# KL annealing schedule validation
# Per CLAUDE.md: beta must ramp 0→0.2 over first 20 epochs
beta_vals = df['beta'].values
print(f'beta[0]  = {beta_vals[0]:.4f}  (expected ~0)')
print(f'beta[19] = {beta_vals[min(19, len(beta_vals)-1)]:.4f}  (expected ~0.2)')
print(f'beta[-1] = {beta_vals[-1]:.4f}  (expected 0.2)')

assert beta_vals[0] < 0.02, 'KL annealing should start near 0'
assert abs(beta_vals[-1] - 0.2) < 0.01, 'KL annealing should plateau at 0.2'
print('\nKL annealing schedule: OK')